In [1]:
import pandas as pd

In [2]:
import pickle

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [4]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("nyc-taxi-experiment")

2026/08/05 01:54:12 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1785894852466, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785894852466, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [5]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [6]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [7]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [8]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [9]:
import xgboost as xgb

In [10]:
from pathlib import Path

In [11]:
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [12]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=30,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/home/ubuntu/anaconda3/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [01:58:53] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.44174
[1]	validation-rmse:10.76627
[2]	validation-rmse:10.17716
[3]	validation-rmse:9.66439
[4]	validation-rmse:9.21865
[5]	validation-rmse:8.83455
[6]	validation-rmse:8.50406
[7]	validation-rmse:8.22010
[8]	validation-rmse:7.97540
[9]	validation-rmse:7.76692
[10]	validation-rmse:7.58908
[11]	validation-rmse:7.43668
[12]	validation-rmse:7.30705
[13]	validation-rmse:7.19678
[14]	validation-rmse:7.10367
[15]	validation-rmse:7.02340
[16]	validation-rmse:6.95494
[17]	validation-rmse:6.89584
[18]	validation-rmse:6.84617
[19]	validation-rmse:6.80219
[20]	validation-rmse:6.76453
[21]	validation-rmse:6.73295
[22]	validation-rmse:6.70496
[23]	validation-rmse:6.68006
[24]	validation-rmse:6.65942
[25]	validation-rmse:6.64052
[26]	validation-rmse:6.62474
[27]	validation-rmse:6.61037
[28]	validation-rmse:6.59772
[29]	validation-rmse:6.58649


2026/08/05 01:59:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run polite-moth-832 at: http://localhost:5000/#/experiments/1/runs/34348857cd7d43a8a8cf4fe71314aed4
🧪 View experiment at: http://localhost:5000/#/experiments/1
